# Build the flight–weather joint table

Run after `clean_flights_weather.ipynb` from the repository root. The current report documents selected months only; set `RUN_YEAR` and `RUN_MONTHS` to choose the month partitions to build.


In [ ]:


from pathlib import Path
from pyspark.sql import SparkSession, functions as F, Window
from datetime import datetime

PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data"

spark = (
    SparkSession.builder
    .appName("Flight Delay Analysis")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

RUN_YEAR = 2013
RUN_MONTHS = [4, 6, 9, 11]

FLIGHT_CLEAN_PATH = str(DATA_ROOT / "clean" / "Flights")
WEATHER_CLEAN_PATH = str(DATA_ROOT / "clean" / "Weather")
JT_OUTPUT_PATH = str(DATA_ROOT / "jt")

# =========================
# Functions
# =========================

def read_flight_month(year: int, month: int):
    path = f"{FLIGHT_CLEAN_PATH}/flight_year={year}/flight_month={month}"
    return spark.read.parquet(path)

def prev_month(year: int, month: int):
    if month == 1:
        return year - 1, 12
    return year, month - 1

def next_month(year: int, month: int):
    if month == 12:
        return year + 1, 1
    return year, month + 1

def weather_partition_path(year: int, month: int):
    return f"{WEATHER_CLEAN_PATH}/weather_year={year}/weather_month={month}"

def path_exists(path: str):
    try:
        spark.read.parquet(path).limit(1).count()
        return True
    except Exception:
        return False

def read_weather_with_buffer(year: int, month: int):
    months = [prev_month(year, month), (year, month), next_month(year, month)]
    dfs = []

    for y, m in months:
        p = weather_partition_path(y, m)
        if path_exists(p):
            dfs.append(spark.read.parquet(p))

    if not dfs:
        raise ValueError(f"No weather parquet found for buffered months around {year}-{month:02d}")

    out = dfs[0]
    for d in dfs[1:]:
        out = out.unionByName(d)
    return out

lags = list(range(13))
lag_df = spark.createDataFrame([(i,) for i in lags], ["lag_hour"])

def make_targets(flights_df, airport_col: str, time_col: str):
    return (
        flights_df
        .select("flight_id", airport_col, time_col)
        .crossJoin(lag_df)
        .withColumnRenamed(airport_col, "airport_id")
        .withColumnRenamed(time_col, "base_ts")
        .withColumn(
            "target_ts",
            F.from_unixtime(
                F.unix_timestamp("base_ts") - F.col("lag_hour") * 3600
            ).cast("timestamp")
        )
    )

def build_weather_long(
    flights_df,
    weather_df,
    airport_col: str,
    time_col: str,
    prefix: str,
    window_hours: int = 1
):
    targets = make_targets(flights_df, airport_col, time_col)

    joined_candidates = (
        targets.alias("f")
        .join(
            weather_df.alias("w"),
            on="airport_id",
            how="left"
        )
        .withColumn(
            "time_diff_sec",
            F.abs(F.unix_timestamp("w.obs_ts") - F.unix_timestamp("f.target_ts"))
        )
        .filter(F.col("time_diff_sec") <= window_hours * 3600)
    )

    w = Window.partitionBy("flight_id", "lag_hour").orderBy(
        F.col("time_diff_sec").asc(),
        F.col("obs_ts").asc()
    )

    out = (
        joined_candidates
        .withColumn("rn", F.row_number().over(w))
        .filter(F.col("rn") == 1)
        .select(
            "flight_id",
            "airport_id",
            "lag_hour",
            "target_ts",
            F.col("obs_ts").alias(f"{prefix}_obs_ts"),
            "time_diff_sec",
            F.col("SkyCondition").alias(f"{prefix}_SkyCondition"),
            F.col("Visibility").alias(f"{prefix}_Visibility"),
            F.col("WeatherType").alias(f"{prefix}_WeatherType"),
            F.col("DryBulbFarenheit").alias(f"{prefix}_DryBulbFarenheit"),
            F.col("WetBulbFarenheit").alias(f"{prefix}_WetBulbFarenheit"),
            F.col("DewPointFarenheit").alias(f"{prefix}_DewPointFarenheit"),
            F.col("RelativeHumidity").alias(f"{prefix}_RelativeHumidity"),
            F.col("WindSpeed").alias(f"{prefix}_WindSpeed"),
            F.col("WindDirection").alias(f"{prefix}_WindDirection"),
            F.col("StationPressure").alias(f"{prefix}_StationPressure"),
            F.col("SeaLevelPressure").alias(f"{prefix}_SeaLevelPressure"),
            F.col("HourlyPrecip").alias(f"{prefix}_HourlyPrecip"),
            F.col("Altimeter").alias(f"{prefix}_Altimeter")
        )
    )

    return out

def long_to_wide(long_df, prefix: str):
    value_cols = [
        c for c in long_df.columns
        if c not in ["flight_id", "airport_id", "lag_hour", "target_ts"]
    ]

    agg_exprs = []
    for lag in range(13):
        for c in value_cols:
            agg_exprs.append(
                F.max(F.when(F.col("lag_hour") == lag, F.col(c))).alias(f"{prefix}{lag}_{c}")
            )

    return long_df.groupBy("flight_id").agg(*agg_exprs)

def build_jt_month(year: int, month: int):
    flights_m = read_flight_month(year, month)
    weather_m = read_weather_with_buffer(year, month)

    flights_keep_cols = [
        "flight_id",
        "FL_DATE",
        "flight_yyyymm",
        "OP_CARRIER_AIRLINE_ID",
        "OP_CARRIER_FL_NUM",
        "ORIGIN_AIRPORT_ID",
        "DEST_AIRPORT_ID",
        "scheduled_dep_ts",
        "scheduled_arr_ts_dest_local",
        "ARR_DELAY_NEW",
        "arr_delay_min",
        "WEATHER_DELAY",
        "NAS_DELAY",
        "has_weather_delay",
        "has_nas_delay",
        "is_delayed_15",
        "is_delayed_60"
    ]

    flights_m = flights_m.select(*flights_keep_cols)

    flight_join_cols = [
        "flight_id",
        "ORIGIN_AIRPORT_ID",
        "DEST_AIRPORT_ID",
        "scheduled_dep_ts",
        "scheduled_arr_ts_dest_local"
    ]

    flights_for_join = flights_m.select(*flight_join_cols)

    weather_for_join = (
        weather_m
        .select(
            F.col("AirportID").alias("airport_id"),
            "obs_ts",
            "SkyCondition",
            "Visibility",
            "WeatherType",
            "DryBulbFarenheit",
            "WetBulbFarenheit",
            "DewPointFarenheit",
            "RelativeHumidity",
            "WindSpeed",
            "WindDirection",
            "StationPressure",
            "SeaLevelPressure",
            "HourlyPrecip",
            "Altimeter"
        )
        .filter(F.col("airport_id").isNotNull())
        .filter(F.col("obs_ts").isNotNull())
    )

    wo_long = build_weather_long(
        flights_df=flights_for_join,
        weather_df=weather_for_join,
        airport_col="ORIGIN_AIRPORT_ID",
        time_col="scheduled_dep_ts",
        prefix="wo",
        window_hours=1
    )

    wd_long = build_weather_long(
        flights_df=flights_for_join,
        weather_df=weather_for_join,
        airport_col="DEST_AIRPORT_ID",
        time_col="scheduled_arr_ts_dest_local",
        prefix="wd",
        window_hours=1
    )

    wo_wide = long_to_wide(wo_long, "wo")
    wd_wide = long_to_wide(wd_long, "wd")

    jt_month = (
        flights_m
        .join(wo_wide, on="flight_id", how="left")
        .join(wd_wide, on="flight_id", how="left")
    )

    return jt_month

# =========================
# Run for selected months
# =========================
for run_month in RUN_MONTHS:
    print(f"Start processing {RUN_YEAR}-{run_month:02d} ...")

    jt_month = build_jt_month(RUN_YEAR, run_month)

    (
        jt_month
        .write
        .mode("overwrite")
        .parquet(f"{JT_OUTPUT_PATH}/year={RUN_YEAR}/month={run_month}")
    )

    print(f"Finished {RUN_YEAR}-{run_month:02d}")